# Bayesian Linear Regression with PyMC

This notebook demonstrates practical Bayesian modelling with PyMC:
1. **Model specification** with priors
2. **Prior predictive checks**
3. **Posterior sampling** with NUTS
4. **Model diagnostics** and comparison

If PyMC is not installed, we fall back to a manual MCMC implementation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
    print(f'PyMC version: {pm.__version__}')
except ImportError:
    HAS_PYMC = False
    print('PyMC not installed. Using manual MCMC fallback.')
    print('Install with: pip install pymc arviz')

%matplotlib inline
np.random.seed(42)

## 1. Generate Synthetic Data

$$y = \beta_0 + \beta_1 x + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

In [ ]:
# True parameters
true_beta0, true_beta1, true_sigma = 1.5, 2.3, 0.8
n = 50

x = np.random.uniform(0, 5, n)
y = true_beta0 + true_beta1 * x + np.random.normal(0, true_sigma, n)

plt.scatter(x, y, alpha=0.7)
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Data (true: y = {true_beta0} + {true_beta1}x, σ = {true_sigma})')
plt.show()

## 2. Bayesian Linear Regression

**Priors:**
- $\beta_0 \sim \mathcal{N}(0, 10)$
- $\beta_1 \sim \mathcal{N}(0, 10)$
- $\sigma \sim \text{HalfNormal}(5)$

In [ ]:
if HAS_PYMC:
    with pm.Model() as linear_model:
        # Priors
        beta0 = pm.Normal('beta0', mu=0, sigma=10)
        beta1 = pm.Normal('beta1', mu=0, sigma=10)
        sigma = pm.HalfNormal('sigma', sigma=5)

        # Likelihood
        mu = beta0 + beta1 * x
        likelihood = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)

        # Prior predictive check
        prior_pred = pm.sample_prior_predictive(samples=200, random_seed=42)

    # Plot prior predictive
    fig, ax = plt.subplots()
    x_sorted = np.sort(x)
    for i in range(50):
        b0 = prior_pred.prior['beta0'].values[0, i]
        b1 = prior_pred.prior['beta1'].values[0, i]
        ax.plot(x_sorted, b0 + b1 * x_sorted, alpha=0.1, color='steelblue')
    ax.scatter(x, y, color='red', zorder=5, s=15)
    ax.set_title('Prior Predictive Check')
    ax.set_ylim(-30, 30)
    plt.show()
else:
    print('Prior predictive check skipped (no PyMC). Will use manual MCMC below.')

In [ ]:
if HAS_PYMC:
    with linear_model:
        trace = pm.sample(2000, tune=1000, cores=1, random_seed=42, return_inferencedata=True)
    az.plot_trace(trace, var_names=['beta0', 'beta1', 'sigma'])
    plt.tight_layout()
    plt.show()
    print(az.summary(trace, var_names=['beta0', 'beta1', 'sigma']))
else:
    # Manual MH for Bayesian linear regression
    def log_posterior_linreg(params):
        b0, b1, log_s = params
        s = np.exp(log_s)
        if s <= 0:
            return -np.inf
        # Priors
        lp = stats.norm.logpdf(b0, 0, 10) + stats.norm.logpdf(b1, 0, 10)
        lp += stats.halfnorm.logpdf(s, scale=5) + log_s  # Jacobian for log transform
        # Likelihood
        mu = b0 + b1 * x
        lp += np.sum(stats.norm.logpdf(y, mu, s))
        return lp

    n_samples = 20000
    samples = np.zeros((n_samples, 3))
    current = np.array([0.0, 0.0, 0.0])
    current_lp = log_posterior_linreg(current)
    prop_std = np.array([0.2, 0.1, 0.1])
    accepted = 0

    for i in range(n_samples):
        proposal = current + np.random.normal(0, prop_std)
        proposal_lp = log_posterior_linreg(proposal)
        if np.log(np.random.rand()) < proposal_lp - current_lp:
            current = proposal
            current_lp = proposal_lp
            accepted += 1
        samples[i] = current

    burn_in = 5000
    samples = samples[burn_in:]
    samples[:, 2] = np.exp(samples[:, 2])  # transform back to sigma

    print(f'Acceptance rate: {accepted/n_samples:.3f}')
    labels = ['beta0', 'beta1', 'sigma']
    truths = [true_beta0, true_beta1, true_sigma]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for i, (ax, lbl, truth) in enumerate(zip(axes, labels, truths)):
        ax.hist(samples[:, i], bins=50, density=True, alpha=0.7)
        ax.axvline(truth, color='red', ls='--', label=f'True={truth}')
        ax.axvline(samples[:, i].mean(), color='orange', ls='--', label=f'Mean={samples[:, i].mean():.2f}')
        ax.set_title(lbl)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 3. Posterior Predictive Check

In [ ]:
x_new = np.linspace(0, 5, 100)

if HAS_PYMC:
    b0_samples = trace.posterior['beta0'].values.flatten()
    b1_samples = trace.posterior['beta1'].values.flatten()
    s_samples = trace.posterior['sigma'].values.flatten()
else:
    b0_samples = samples[:, 0]
    b1_samples = samples[:, 1]
    s_samples = samples[:, 2]

fig, ax = plt.subplots(figsize=(8, 5))
# Draw posterior regression lines
idx = np.random.choice(len(b0_samples), 200, replace=False)
for i in idx:
    ax.plot(x_new, b0_samples[i] + b1_samples[i] * x_new, alpha=0.02, color='steelblue')

# Mean prediction
mean_line = b0_samples.mean() + b1_samples.mean() * x_new
ax.plot(x_new, mean_line, 'r-', lw=2, label='Posterior mean')
ax.scatter(x, y, color='black', s=20, zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Posterior Predictive: Uncertainty in Regression Line')
ax.legend()
plt.show()

## Key Takeaways

- **PyMC** provides a high-level API for specifying Bayesian models.
- **Prior predictive checks** ensure your priors are sensible before fitting.
- **NUTS** (No U-Turn Sampler) is far more efficient than basic Metropolis-Hastings.
- **Posterior predictive checks** visualise uncertainty in model predictions.
- Always inspect **trace plots**, **R-hat**, and **ESS** diagnostics.

**Next:** Hierarchical models and the power of partial pooling.